### Content: data cleaning
- remove wrong coordinates (latitude or longitude less than zero, in particular some rows have values equal to -180)
- fix point_trip_id: some ID repeat themselves after some days: must modify them so that each ID is unique for each trip (different ID in different days)

Produce *trips_pointv3_cleaned.parquet*.

### Issue 1: filter out wrong coordinates

In [1]:
import pandas as pd

In [2]:
data = pd.read_parquet('trips_pointv3.parquet')
data

,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability
0,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:46:54+00:00,46.052974,11.120415,13,None
1,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:04+00:00,46.052951,11.120469,14,None
2,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:15+00:00,46.052905,11.120811,15,None
3,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:24+00:00,46.052874,11.121122,16,None
4,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:35+00:00,46.052732,11.121525,17,None
...,...,...,...,...,...,...,...,...
999995,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:06:41+00:00,46.079951,11.117475,3,None
999996,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:06:51+00:00,46.079677,11.117734,4,None
999997,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:07:01+00:00,46.079403,11.117844,5,None
999998,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:07:11+00:00,46.079176,11.117913,6,None


In [3]:
data.point_reliability.unique() # all None

array([None], dtype=object)

In [4]:
data[data.point_latitude <= 0]

,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability
475591,VENTO,WIND,900475a38fe9,2021-10-26 15:19:32+00:00,-180.000000,-180.000000,8,None
555862,VENTO,WIND,5e77bb1ac8cc,2021-06-08 19:56:41+00:00,-180.000000,-180.000000,25,None
888941,VENTO,WIND,8881fc856210,2021-05-04 16:06:22+00:00,-180.000000,-180.000000,39,None


In [5]:
data[data.point_longitude <= 0]

,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability
475591,VENTO,WIND,900475a38fe9,2021-10-26 15:19:32+00:00,-180.000000,-180.000000,8,None
555862,VENTO,WIND,5e77bb1ac8cc,2021-06-08 19:56:41+00:00,-180.000000,-180.000000,25,None
888941,VENTO,WIND,8881fc856210,2021-05-04 16:06:22+00:00,-180.000000,-180.000000,39,None


In [6]:
data = data[data.point_longitude >= 0]
data = data[data.point_latitude >= 0]
data

,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability
0,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:46:54+00:00,46.052974,11.120415,13,None
1,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:04+00:00,46.052951,11.120469,14,None
2,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:15+00:00,46.052905,11.120811,15,None
3,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:24+00:00,46.052874,11.121122,16,None
4,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:35+00:00,46.052732,11.121525,17,None
...,...,...,...,...,...,...,...,...
999995,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:06:41+00:00,46.079951,11.117475,3,None
999996,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:06:51+00:00,46.079677,11.117734,4,None
999997,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:07:01+00:00,46.079403,11.117844,5,None
999998,VENTO,Tier,66cff91f-babc-4560-b5c3-a6cc53c6d371,2022-03-04 13:07:11+00:00,46.079176,11.117913,6,None


In [7]:
data = data.reset_index()

### Issue 2: ID repeat themselves in different days

In [8]:
data[data.point_trip_id =='00000001-0000-1000-9ec1-5b7d6a617b60'] ## how is it that the same ID appears in different days? does it mean it's the same person?

,index,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability
80744,80744,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2020-12-03 14:14:57+00:00,46.060532,11.115307,0,None
80974,80974,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2020-12-03 14:44:40+00:00,46.060551,11.115251,1,None
93023,93023,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-02-16 15:05:28+00:00,46.071957,11.120046,0,None
97117,97117,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-02-16 15:15:17+00:00,46.060024,11.133364,1,None
101247,101247,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-03-20 20:52:02+00:00,46.061253,11.126844,1,None
102618,102618,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-03-20 20:14:08+00:00,46.061020,11.124775,0,None
112022,112022,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-05-04 14:19:43+00:00,46.073273,11.126663,0,None
116615,116615,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-05-04 14:24:13+00:00,46.082153,11.122440,1,None
119678,119678,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-05-19 17:48:56+00:00,46.083977,11.119506,0,None
119679,119679,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2021-05-19 17:55:59+00:00,46.069721,11.121650,1,None


In [9]:
# add date column
lst_gg = []
for gg in range(len(data.point_timestamp)):
  lst_gg.append(data.point_timestamp[gg].date())
data['date'] = lst_gg

In [10]:
# create a unique id as concatenation of point_trip_id and date
id=[]
for r in range(len(data)):
  id.append(str(data.point_trip_id[r])+'_'+str(data.date[r]))
data['unique_id'] = id

In [11]:
data.head()

,index,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability,date,unique_id
0,0,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:46:54+00:00,46.052974,11.120415,13,None,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07
1,1,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:04+00:00,46.052951,11.120469,14,None,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07
2,2,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:15+00:00,46.052905,11.120811,15,None,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07
3,3,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:24+00:00,46.052874,11.121122,16,None,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07
4,4,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:35+00:00,46.052732,11.121525,17,None,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07


In [12]:
data[data.unique_id == '00000001-0000-1000-9ec1-5b7d6a617b60_2020-12-03']

,index,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,point_reliability,date,unique_id
80744,80744,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2020-12-03 14:14:57+00:00,46.060532,11.115307,0,None,2020-12-03,00000001-0000-1000-9ec1-5b7d6a617b60_2020-12-03
80974,80974,BIT,Bit Mobility,00000001-0000-1000-9ec1-5b7d6a617b60,2020-12-03 14:44:40+00:00,46.060551,11.115251,1,None,2020-12-03,00000001-0000-1000-9ec1-5b7d6a617b60_2020-12-03


In [13]:
data = data.drop(['index', 'point_reliability'], axis=1) # drop also "point_reliability" because filled with None only
data.head(2)

,operator,point_operator_id,point_trip_id,point_timestamp,point_latitude,point_longitude,point_sequence,date,unique_id
0,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:46:54+00:00,46.052974,11.120415,13,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07
1,VENTO,Tier,24c83c4e-1e60-4b65-9100-8cf1550076df,2022-01-07 20:47:04+00:00,46.052951,11.120469,14,2022-01-07,24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07


Save cleaned file.

In [14]:
import os
import sys 
from google.colab import drive 
import glob

path = "/content/drive" 
drive.mount(path, force_remount=True)  
data_path = "drive/MyDrive/Colab Notebooks/escooter_TN" 

myfiles = glob.glob(os.path.join(data_path, '*')) 
print(myfiles)

Mounted at /content/drive
['drive/MyDrive/Colab Notebooks/escooter_TN/trento.pbf', 'drive/MyDrive/Colab Notebooks/escooter_TN/dataviz_spostamenti.html', 'drive/MyDrive/Colab Notebooks/escooter_TN/dataviz_spostamenti_monopattini.ipynb', 'drive/MyDrive/Colab Notebooks/escooter_TN/traiettorie_con_MovingPandas.ipynb', 'drive/MyDrive/Colab Notebooks/escooter_TN/check_datasets-delete.ipynb', 'drive/MyDrive/Colab Notebooks/escooter_TN/traiettorie.ipynb', 'drive/MyDrive/Colab Notebooks/escooter_TN/trips_pointv3_cleaned.parquet', 'drive/MyDrive/Colab Notebooks/escooter_TN/ExploreTrips.ipynb', 'drive/MyDrive/Colab Notebooks/escooter_TN/0_clean_data.ipynb']


In [15]:
data.to_parquet(data_path+'/trips_pointv3_cleaned.parquet')